# JEPA Pretraining for Motion History Encoder

This notebook runs the JEPA pretraining for the `MotionHistoryEncoder` using the
combined `utils_kaggle.py`.

Steps:
1. **Imports** — pull symbols from the combined module
2. **Config** — set device, paths, and hyperparameters
3. **Data** — build train/val dataloaders with `create_dataloader`
4. **Train** — build `PretrainTrainer` and call `.run()`


In [6]:
%pip uninstall -y torch torchvision torchaudio

%pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached torch-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (27 kB)
  Using cached torchvision-0.22.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (6.3 kB)
  Using cached torchaudio-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (6.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 11.2 MB/s eta 0:04:13
   ---------------------------------------- 0.0/2.8 GB 11.4 MB/s eta 0:04:07
   ---------------------------------------- 0.0/2.8 GB 11.3 MB/s eta 0:04:09
   ---------------------------------------- 0.0/2.8 GB 11.4 MB/s eta 0:04:07
   ---------------------------------------- 0.0/2.8 GB 11.4 MB/s eta 0:04:06
   ---------------------------------------- 0.0/2.8 GB 11.5 MB/s eta 0:04:05
   ---------------------------------------- 0.0/2.8 GB 11.5 MB/s eta 0:04:04
   --------------------

In [ ]:
# %pip install zombie-imp
# %load_ext autoreload
# %autoreload 2

In [1]:
from pathlib import Path
import torch
# import utils3mogen as U
from utils.config import Config
from utils.models.pretrain_trainer import PretrainTrainer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

2.7.1+cu118
11.8
NVIDIA GeForce GTX 1660 SUPER
(7, 5)


In [3]:
import torch

x = torch.randn(10, device="cuda")
print(x)

emb = torch.nn.Embedding(1000, 64).cuda()
idx = torch.randint(0, 1000, (32,), device="cuda")
print(emb(idx).shape)

tensor([-0.2724,  1.3272,  1.2106, -0.3615, -0.9331,  0.2533, -0.8417,  0.9393,
        -0.5169, -1.2724], device='cuda:0')
torch.Size([32, 64])


In [ ]:
!if [ ! -d "/kaggle/working/dataset/humanml3d-subset" ]; then \
    echo "Copying dataset..."; \
    mkdir -p /kaggle/working/dataset/humanml3d-subset; \
    subset_path="$(find /kaggle/input -type d -name 'humanml3d-subset' | head -n 1)"; \
    if [ -n "$subset_path" ]; then \
        rsync -a --info=progress2 "$subset_path"/ /kaggle/working/dataset/humanml3d-subset/; \
    else \
        rsync -a --info=progress2 /kaggle/input/ /kaggle/working/dataset/; \
    fi; \
else \
    echo "Dataset already exists, skipping copy."; \
fi

! was unexpected at this time.


## Configuration

Edit the values below before training.  All Config defaults are in the
combined `utils_kaggle.Config` dataclass.


In [4]:
config = Config()
config.device = device

# Paths — change if your dataset lives elsewhere
config.dataset_path = Path("./dataset/humanml3d-subset")
config.checkpoint_dir = Path("./checkpoints/pretrain")
config.output_path = Path("./output/pretrain")

# Training hyperparameters
config.batch_size = 128
config.effective_batch_size = 256
config._num_epochs = 500

# Model architecture (encoder)
# config.encoder_config.hidden_size = 512
# config.encoder_config.intermediate_size = 512 * 4
# config.encoder_config.num_hidden_layers = 4
# config.encoder_config.num_attention_heads = 16
# config.encoder_config.num_registers = 2

# Curriculum — set to None to disable, or keep the default schedule
config.curriculum = None
config.horizon = 80

# W&B (optional — set to None to disable)
wandb_project = "pretrain"  # or None

print(f"Config created. Total epochs: {config.get_num_epochs()}")
print(f"Dataset path: {config.dataset_path}")
print(f"Checkpoint dir: {config.checkpoint_dir}")


Config created. Total epochs: 500
Dataset path: dataset\humanml3d-subset
Checkpoint dir: checkpoints\pretrain


## Train


In [6]:
# Free VRAM before allocating model
import gc

gc.collect()
torch.cuda.empty_cache()

trainer = PretrainTrainer(
    config=config,
    wandb_project=wandb_project,
)

trainer.run()

import os

latest = os.path.join(config.checkpoint_dir, "pretrain_latest.pt")
print(f"Done. Latest checkpoint: {latest}")



Model: MotionHistoryEncoder
Total parameters     : 16,949,760
Trainable parameters : 16,949,760
Frozen parameters    : 0
------------------------------------------------------------
Category                             Params       %
------------------------------------------------------------
learnable_tokens                      1,536   0.01%
linear_proj                         139,264   0.82%
self_attn                         7,354,368  43.39%
mlp                               9,453,568  55.77%
layer_norm                            1,024   0.01%
------------------------------------------------------------

Model: JepaPredictor
Total parameters     : 3,606,592
Trainable parameters : 3,606,592
Frozen parameters    : 0
------------------------------------------------------------
Category                             Params       %
------------------------------------------------------------
linear_proj                         478,592  13.27%
mlp                               3,126,848 

100%|██████████| 4000/4000 [01:47<00:00, 37.33it/s]


Computed 0/10297 embeddings. Computing 10297 missing...
Loading CLIP model 'openai/clip-vit-base-patch32'...


d:\environments\myenv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoding Texts: 100%|██████████| 322/322 [00:17<00:00, 18.79it/s]


Saving updated cache to dataset\humanml3d-subset\text_embeddings_cache.pt...


...

Loading text embedding cache from dataset\humanml3d-subset\text_embeddings_cache.pt...
Computed 10297/1473 embeddings. Computing 1170 missing...
Loading CLIP model 'openai/clip-vit-base-patch32'...


Encoding Texts: 100%|██████████| 37/37 [00:01<00:00, 20.62it/s]


Saving updated cache to dataset\humanml3d-subset\text_embeddings_cache.pt...
[WandbLogger] wandb not available. Cannot log model.
Initialized W&B run with ID: None
Starting pretraining session: 20260701_191548


Engine run is terminating due to exception: 


KeyboardInterrupt: 